# Airbnb Price Prediction — Modular Pipeline
Each module below is **fully independent**: it saves all outputs to Drive and loads what it needs at the top.
Run only the modules you need — no need to re-run upstream cells.

```
MODULE 0 — Setup & GPU install        (run once per Colab session)
MODULE 1 — Sentiment Analysis         saves: reviews_sentiment.csv
MODULE 2 — Preprocessing & Split      saves: data_cleaned.csv, split CSVs, scaler.pkl
MODULE 3 — Feature Selection          saves: selected_features_lasso.npy, selected_features_pvals.npy, feature_selection_r2.json
MODULE 4 — Model Training             saves: results_metrics.csv, best_params.json, trained_models/ (joblib)
MODULE 5 — SHAP Interpretability      loads: trained models + lasso features
MODULE 6 — Variance Assessment        saves: variance_results.csv
MODULE 7 — Architectural Exploration  saves: arch_results.csv
```

## MODULE 0 — Setup & GPU Install
Run this once at the start of each Colab session to install dependencies and mount Drive.

⚠️ DATA_DIR and MODELS_DIR need to be updated according to your own drive folders !

In [ ]:
import sys
import subprocess, threading

install_done = threading.Event()


def install_cuml():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--extra-index-url",
            "https://pypi.nvidia.com",
            "cuml-cu12==25.2.*",
            "-q",
        ],
    )
    install_done.set()


threading.Thread(target=install_cuml, daemon=True).start()
print("cuML install started in background...")

In [ ]:
from textblob import TextBlob
import pandas as pd
import math
import os, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt

# from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import feature_selection
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import r2_score

import multiprocessing
import sklearn as sklearn
from sklearn import linear_model
from sklearn import metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, Lasso, SGDRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor

# Suppress some common sklearn warnings
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings(
    "ignore", category=UserWarning
)  # For KMeans n_init being deprecated

from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
!pip install -q optuna lightgbm shap
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/ML_Practice/Data"
MODELS_DIR = "/content/drive/MyDrive/Colab Notebooks/ML_Practice/Models"
import os

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("✅ Drive mounted")
print(f"   DATA_DIR   = {DATA_DIR}")
print(f"   MODELS_DIR = {MODELS_DIR}")

---
## MODULE 1 — Sentiment Analysis
**Inputs:** `reviews_original.csv` in DATA_DIR  
**Outputs:** `reviews_sentiment.csv`  
Run once; skip if `reviews_sentiment.csv` already exists.

In [ ]:
# ── Sentinel: skip if already done ──────────────────────────────────────────
import os, pandas as pd

_out = os.path.join(DATA_DIR, "reviews_sentiment.csv")
if os.path.exists(_out):
    print(f"✅ Already exists — skipping. ({_out})")
else:
    from textblob import TextBlob
    import numpy as np

    filename = os.path.join(DATA_DIR, "reviews_original.csv")
    data = pd.read_csv(filename)
    data = data.drop(
        columns=["id", "date", "reviewer_id", "reviewer_name"], errors="ignore"
    )

    def calculate_sentiment(entry):
        if not isinstance(entry, str):
            return 0.0
        return TextBlob(entry).sentiment.polarity

    data["sentiment"] = data["comments"].apply(calculate_sentiment)
    avg_sentiment = data.groupby("listing_id")["sentiment"].mean().reset_index()
    avg_sentiment.columns = ["listing_id", "avg_sentiment"]
    avg_sentiment.to_csv(_out, index=False)
    print(f"✅ Saved → {_out}")

---
## MODULE 2 — Preprocessing, Splitting & Normalisation
**Inputs:** `listings.csv`, `reviews_sentiment.csv`  
**Outputs:** `data_cleaned.csv`, `data_cleaned_train_X.csv`, `data_cleaned_train_y.csv`,
`data_cleaned_val_X.csv`, `data_cleaned_val_y.csv`, `data_cleaned_test_X.csv`,
`data_cleaned_test_y.csv`, `scaler.pkl`  
Run once; skip if all outputs already exist.

In [ ]:
# ── Module 2 : Preprocessing ─────────────────────────────────────────────────
import os, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import joblib
import datetime as dt

_required = [
    os.path.join(DATA_DIR, f)
    for f in [
        "data_cleaned_train_X.csv",
        "data_cleaned_train_y.csv",
        "data_cleaned_val_X.csv",
        "data_cleaned_val_y.csv",
        "data_cleaned_test_X.csv",
        "data_cleaned_test_y.csv",
    ]
]

if all(os.path.exists(p) for p in _required):
    print("✅ Files already exist — loading directly.")
    X_train = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_train_X.csv"))
    y_train = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_train_y.csv"))
    X_val = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_val_X.csv"))
    y_val = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_val_y.csv"))
    X_test = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_test_X.csv"))
    y_test = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_test_y.csv"))
    print(f"   X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

else:
    print("Running preprocessing...")

    # ── Load raw data ────────────────────────────────────────────────────────
    listings = pd.read_csv(os.path.join(DATA_DIR, "listings.csv"), low_memory=False)
    sentiment = pd.read_csv(os.path.join(DATA_DIR, "reviews_sentiment.csv"))
    data = listings.merge(sentiment, left_on="id", right_on="listing_id", how="left")

    # ------------------------------------------------------------------
    # Drop unwanted columns
    # ------------------------------------------------------------------
    # These columns are identified as irrelevant, uninformative, or having too many missing values
    data = pd.DataFrame.drop(
        data,
        columns=[
            "host_name",
            "notes",
            "host_about",
            "calendar_updated",
            "host_acceptance_rate",
            "description",
            "thumbnail_url",
            "experiences_offered",
            "listing_url",
            "name",
            "summary",
            "space",
            "scrape_id",
            "last_scraped",
            "neighborhood_overview",
            "transit",
            "access",
            "interaction",
            "house_rules",
            "medium_url",
            "picture_url",
            "xl_picture_url",
            "host_url",
            "host_thumbnail_url",
            "host_picture_url",
            "smart_location",
            "license",
            "jurisdiction_names",
            "street",
            "neighbourhood",
            "country",
            "country_code",
            "host_location",
            "host_neighbourhood",
            "market",
            "is_location_exact",
            "square_feet",
            "weekly_price",
            "monthly_price",
            "availability_30",
            "availability_60",
            "availability_90",
            "availability_365",
            "calendar_last_scraped",
            "first_review",
            "last_review",
            "requires_license",
            "calculated_host_listings_count",
            "host_listings_count",
            "zipcode",
        ],
        errors="ignore",  # Use errors='ignore' for robustness if some columns are already missing
    )

    # ------------------------------------------------------------------
    # Host verifications (convert to binary features)
    # ------------------------------------------------------------------
    print("Splitting host verifications...")
    host_verification_set = set()

    def collect_host_verifications(entry):
        if not isinstance(entry, str):
            return
        entry_list = (
            entry.replace("[", "")
            .replace("]", "")
            .replace("'", "")
            .replace('"', "")
            .replace(" ", "")
            .split(",")
        )
        for verification in entry_list:
            if verification != "" and verification != "None":
                host_verification_set.add(verification + "_verification")

    data["host_verifications"].apply(collect_host_verifications)

    def generic_verification(entry, v):
        entry_list = (
            str(entry)
            .replace("[", "")
            .replace("]", "")
            .replace("'", "")
            .replace('"', "")
            .replace(" ", "")
            .split(",")
        )
        for verification in entry_list:
            if verification + "_verification" == v:
                return 1
        return 0

    for v in host_verification_set:
        data[v] = data["host_verifications"].apply(lambda x: generic_verification(x, v))

    data = pd.DataFrame.drop(data, columns=["host_verifications"], errors="ignore")

    # ------------------------------------------------------------------
    # Cleaning helper functions
    # ------------------------------------------------------------------
    def clean_response_rate(entry):
        if isinstance(entry, str):
            return entry.replace("%", "")
        return 0  # Default to 0 for missing/non-string values

    def clean_superhost(entry):
        return 1 if entry == "t" else 0

    def clean_price(entry):
        if pd.isna(entry):
            return np.nan
        if isinstance(entry, str):
            entry = entry.replace("$", "").replace(",", "")
        try:
            value = float(entry)
            if value <= 0:
                return np.nan  # Prices should be positive
            return np.log(value)  # Apply logarithm to mitigate outliers
        except Exception:
            return np.nan

    def clean_number(entry):
        if pd.isna(entry):
            return 0
        return entry

    # Keep the paper spirit: rows without core room info are removed
    def clean_number_removal(entry):
        if pd.isna(entry):
            return np.nan
        return entry

    # Apply cleaning functions
    if "host_response_rate" in data.columns:
        data["host_response_rate"] = (
            data["host_response_rate"].apply(clean_response_rate).astype(float)
        )

    for col in [
        "host_is_superhost",
        "host_has_profile_pic",
        "host_identity_verified",
        "has_availability",
        "instant_bookable",
        "is_business_travel_ready",
        "require_guest_profile_picture",
        "require_guest_phone_verification",
    ]:
        if col in data.columns:
            data[col] = data[col].apply(clean_superhost)

    # ------------------------------------------------------------------
    # Core numeric columns (handle missing values)
    # ------------------------------------------------------------------
    # Drop rows where core room info (bathrooms, bedrooms, beds) is missing
    for col in ["bathrooms", "bedrooms", "beds"]:
        if col in data.columns:
            data[col] = data[col].apply(clean_number_removal)

    data = data.dropna(subset=["bathrooms", "bedrooms", "beds"])

    # Handle missing values for other numeric columns
    if "reviews_per_month" in data.columns:
        data["reviews_per_month"] = data["reviews_per_month"].apply(clean_number)

    # Clean and log transform the 'price' column
    if "price" in data.columns:
        data["price"] = data["price"].apply(clean_price)

    data = data.dropna(
        subset=["price"]
    )  # Drop rows where price could not be cleaned/logged

    # Clean other price-related columns
    for col in ["extra_people", "security_deposit", "cleaning_fee"]:
        if col in data.columns:
            data[col] = data[col].apply(clean_price)

    # Fill missing fees/deposits with 0 (as per problem description's spirit)
    for col in ["security_deposit", "cleaning_fee"]:
        if col in data.columns:
            data[col] = data[col].fillna(0)

    # Extra people can also reasonably default to 0 if missing
    if "extra_people" in data.columns:
        data["extra_people"] = data["extra_people"].fillna(0)

    # Fill missing host_total_listings_count with 1 (assuming a single listing if not specified)
    if "host_total_listings_count" in data.columns:
        data["host_total_listings_count"] = data["host_total_listings_count"].fillna(1)

    # ------------------------------------------------------------------
    # State cleaning and filtering (specific to 'NY')
    # ------------------------------------------------------------------
    print("Cleaning the state and filtering for NY...")

    def cleaned_state(entry):
        if isinstance(entry, str):
            entry_clean = entry.strip().upper()
            if entry_clean in ["NY", "NEW YORK"]:
                return "NY"
            return entry_clean
        if pd.isna(entry):
            return ""
        return entry

    if "state" in data.columns:
        data["state"] = data["state"].apply(cleaned_state)
        data = data[data["state"] == "NY"]
    else:
        print("Warning: 'state' column not found, skipping state filtering.")

    # ------------------------------------------------------------------
    # Amenities (convert to binary features)
    # ------------------------------------------------------------------
    print("Splitting amenities...")
    amenities_set = set()

    def collect_amenities(entry):
        if not isinstance(entry, str):
            return
        entry_list = (
            entry.replace("{", "")
            .replace("}", "")
            .replace("'", "")
            .replace('"', "")
            .replace(" ", "_")
            .split(",")
        )
        for am in entry_list:
            if "translation_missing" not in am and am != "":
                amenities_set.add(am)

    if "amenities" in data.columns:
        data["amenities"].apply(collect_amenities)

    def generic_amenities(entry, amenity):
        if not isinstance(entry, str):
            return 0
        entry_list = (
            entry.replace("{", "")
            .replace("}", "")
            .replace("'", "")
            .replace('"', "")
            .replace(" ", "_")
            .split(",")
        )
        for am in entry_list:
            if am == amenity:
                return 1
        return 0

    for amenity in amenities_set:
        data[amenity] = data["amenities"].apply(lambda x: generic_amenities(x, amenity))

    data = pd.DataFrame.drop(data, columns=["amenities", "state"], errors="ignore")

    # ------------------------------------------------------------------
    # One-hot encoding of categorical features
    # ------------------------------------------------------------------
    print("Applying one-hot encoding...")
    for col_name in [
        "property_type",
        "bed_type",
        "room_type",
        "neighbourhood_group_cleansed",
        "city",
        "cancellation_policy",
        "host_response_time",
        "neighbourhood_cleansed",
    ]:
        if col_name in data.columns:
            parsed_cols = pd.get_dummies(
                data[col_name], prefix=col_name, dummy_na=False
            ).astype(
                int
            )  # pandas>=2.0 returns bool; cast to int for cuML/XGBoost compatibility  # Add prefix to avoid name clashes
            data = data.drop(columns=[col_name])
            data = pd.concat([data, parsed_cols], axis=1)
        else:
            print(
                f"Warning: Categorical column '{col_name}' not found for one-hot encoding."
            )

    # ------------------------------------------------------------------
    # host_since (convert to days since a dummy date)
    # ------------------------------------------------------------------
    print("Processing host_since...")

    def clean_host_since(entry):
        if pd.isna(entry):
            return np.nan
        return entry

    if "host_since" in data.columns:
        data["host_since"] = data["host_since"].apply(clean_host_since)
        data["host_since"] = pd.to_datetime(data["host_since"], errors="coerce")
        data = data.dropna(
            subset=["host_since"]
        )  # Drop rows where host_since is missing after conversion

        # Calculate days since a dummy date (e.g., a fixed reference date)
        dummy_date = dt.datetime(2018, 11, 10)  # Using the date from the original code
        data["host_since"] = (dummy_date - data["host_since"]).dt.days.astype(float)
    else:
        print("Warning: 'host_since' column not found.")

    # ------------------------------------------------------------------
    # Review scores (fill missing with 0)
    # ------------------------------------------------------------------
    print("Filling missing review scores with 0...")
    for col_name in [
        "review_scores_rating",
        "review_scores_accuracy",
        "review_scores_cleanliness",
        "review_scores_checkin",
        "review_scores_communication",
        "review_scores_location",
        "review_scores_value",
    ]:
        if col_name in data.columns:
            data[col_name] = data[col_name].fillna(0)
        else:
            print(f"Warning: Review score column '{col_name}' not found.")

    # ------------------------------------------------------------------
    # Join comments (sentiment scores) and handle missing
    # ------------------------------------------------------------------
    print("Joining sentiment data...")
    # Ensure 'id' is in data and 'listing_id' in reviews for joining
    print("Joining sentiment data...")
    if "id" in data.columns and "listing_id" in sentiment.columns:
        data = data.set_index("id").join(sentiment.set_index("listing_id"))
    else:
        print("Warning: 'id' or 'listing_id' column missing, skipping sentiment join.")

    def clean_comments(entry):
        if pd.isna(entry):
            return 0  # If sentiment is missing after join, set to 0
        return entry

    if "avg_sentiment" in data.columns:
        data["avg_sentiment"] = data["avg_sentiment"].apply(clean_comments)
    else:
        print("Warning: 'comments' (sentiment) column not found after join.")

    # ------------------------------------------------------------------
    # Final missing-value patch
    # This is crucial for ensuring no NaNs remain before model training
    # ------------------------------------------------------------------

    # Diagnose duplicate columns (if any arise from one-hot encoding or other steps)
    dup_cols = data.columns[data.columns.duplicated()]
    print(
        "\nDuplicate columns found (will remove duplicates, keeping first):",
        list(dup_cols),
    )

    # Remove duplicate columns, keep first occurrence
    if len(dup_cols) > 0:
        data = data.loc[:, ~data.columns.duplicated()].copy()

    # Fill remaining numeric NaN with 0
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        data[col] = data[col].fillna(0)

    # Fill remaining object NaN with empty string
    object_cols = data.select_dtypes(include=[object]).columns.tolist()
    for col in object_cols:
        data[col] = data[col].fillna("")

    # Final diagnostic for missing values
    missing = data.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("\nRemaining missing values after final patch:")
    print(missing if len(missing) > 0 else "No missing values left.")

    print("\nShape after cleaning and feature engineering:", data.shape)

    # Save the cleaned data for the next step
    cleaned_output_path = os.path.join(DATA_DIR, "data_cleaned.csv")
    data.to_csv(cleaned_output_path, index=False)
    print(f"Cleaned data saved to: {cleaned_output_path}")

    # ── Remove rows with NaN price (unfixable) ────────────────────────────────
    data = data.dropna(subset=["price"])
    data = data.reset_index(drop=True)
    print(f"Dataset shape after cleaning: {data.shape}")
    data.to_csv(os.path.join(DATA_DIR, "data_cleaned.csv"), index=False)
    print("✅ data_cleaned.csv saved")

In [ ]:
# ── Split & Normalise (run only if sentinel didn't skip) ─────────────────────
import os, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import joblib

os.makedirs(MODELS_DIR, exist_ok=True)

dataset_cleaned = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned.csv"))


def split(dataset, val_frac=0.05, test_frac=0.05, random_state=1):
    X = dataset.drop(
        columns=[
            c for c in ["price", "id", "host_id", "Unnamed: 0"] if c in dataset.columns
        ]
    )
    y = dataset["price"]
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=val_frac + test_frac, random_state=random_state
    )
    X_test, X_val, y_test, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_frac / (val_frac + test_frac),
        random_state=random_state,
    )
    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = split(dataset_cleaned)

# Fit scaler ONLY on train — no leakage
scaler = preprocessing.MinMaxScaler()
X_train_norm = pd.DataFrame(
    scaler.fit_transform(X_train.astype(float)), columns=X_train.columns
)
X_val_norm = pd.DataFrame(scaler.transform(X_val.astype(float)), columns=X_val.columns)
X_test_norm = pd.DataFrame(
    scaler.transform(X_test.astype(float)), columns=X_test.columns
)

# Save scaler
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.pkl"))

# Save splits
for name, arr in [
    ("data_cleaned_train_X", X_train_norm),
    ("data_cleaned_train_y", y_train),
    ("data_cleaned_val_X", X_val_norm),
    ("data_cleaned_val_y", y_val),
    ("data_cleaned_test_X", X_test_norm),
    ("data_cleaned_test_y", y_test),
]:
    arr.to_csv(os.path.join(DATA_DIR, f"{name}.csv"), index=False)

print(
    f"✅ Splits saved. Train={X_train_norm.shape}, Val={X_val_norm.shape}, Test={X_test_norm.shape}"
)

---
## MODULE 3 — Feature Selection
**Inputs:** split CSVs from MODULE 2  
**Outputs:** `selected_features_lasso.npy`, `selected_features_pvals.npy`, `feature_selection_r2.json`  
Run once; skip if outputs already exist.

In [ ]:
# Sentinel: skip if already done
import os, numpy as np, json

_lasso_path = os.path.join(DATA_DIR, "selected_features_lasso.npy")
_pvals_path = os.path.join(DATA_DIR, "selected_features_pvals.npy")
_r2_path = os.path.join(DATA_DIR, "feature_selection_r2.json")

if all(os.path.exists(p) for p in [_lasso_path, _pvals_path, _r2_path]):
    print("✅ Feature selection outputs already exist — skipping.")
    selected_features_lasso = np.load(_lasso_path, allow_pickle=True).tolist()
    selected_features_pvals = np.load(_pvals_path, allow_pickle=True).tolist()
    print(
        f"   Lasso features: {len(selected_features_lasso)} | P-val features: {len(selected_features_pvals)}"
    )
else:
    # File paths for input and output data
    train_X_filename = os.path.join(DATA_DIR, "data_cleaned_train_X.csv")
    train_y_filename = os.path.join(DATA_DIR, "data_cleaned_train_y.csv")
    val_X_filename = os.path.join(DATA_DIR, "data_cleaned_val_X.csv")
    val_y_filename = os.path.join(DATA_DIR, "data_cleaned_val_y.csv")

    # Load the training and validation data
    X_train = pd.read_csv(train_X_filename)
    y_train = pd.read_csv(train_y_filename)

    X_val = pd.read_csv(val_X_filename)
    y_val = pd.read_csv(val_y_filename)

    print("Data loaded successfully.")
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

    # Ensure y_train and y_val are 1-D arrays for f_regression and model training
    y_train = y_train.iloc[:, 0]  # Assuming single target column
    y_val = y_val.iloc[:, 0]  # Assuming single target column

    def adjusted_r2(y_true, y_pred, n_features):
        n = len(y_true)
        r2 = r2_score(y_true, y_pred)
        return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

    # ------------------------------------------------------------------
    # 1. Feature Selection using p-values (refined as per paper)
    #    - Selects up to 100 features based on lowest p-values.
    #    - The final set is the one that gives the best R2 on validation.
    # ------------------------------------------------------------------
    print("\n--- P-value based Feature Selection ---")
    F_vals, p_vals = feature_selection.f_regression(X_train, y_train)

    # Create a DataFrame for p-values to easily sort and select
    p_value_df = pd.DataFrame({"feature": X_train.columns, "p_value": p_vals})

    # Handle NaN p-values by replacing them with 1 (no significance)
    p_value_df["p_value"] = p_value_df["p_value"].fillna(1)

    # Sort features by p-value in ascending order
    p_value_df = p_value_df.sort_values(by="p_value", ascending=True).reset_index(
        drop=True
    )

    best_r2_pvals = -np.inf
    best_k_pvals = 0
    selected_features_pvals = []

    # Loop from 1 to 100 features (or max available if less than 100)
    # to find the optimal number of features that yield the best R2 on validation
    max_features_to_check = min(100, len(X_train.columns))

    for k in range(1, max_features_to_check + 1):
        current_p_features = p_value_df["feature"].head(k).tolist()

        # Train a Linear Regression model with the current set of features
        model = LinearRegression()
        model.fit(X_train[current_p_features], y_train)

        # Evaluate R2 on the validation set
        y_val_pred = model.predict(X_val[current_p_features])
        # current_r2 = r2_score(y_val, y_val_pred)
        current_r2 = adjusted_r2(y_val, y_val_pred, k)

        if current_r2 > best_r2_pvals:
            best_r2_pvals = current_r2
            best_k_pvals = k
            selected_features_pvals = current_p_features
        print(f"  k={k}, R2={current_r2:.4f}")  # Uncomment for detailed progress

    print(
        f"Best R2 for p-value selection on validation: {best_r2_pvals:.4f} with {best_k_pvals} features."
    )
    print(f"Number of p-value selected features: {len(selected_features_pvals)}")
    print("Selected p-value features (first 5):", selected_features_pvals[:5])

    # Save the selected features
    output_path_pvals = os.path.join(DATA_DIR, "selected_features_pvals.npy")
    np.save(output_path_pvals, selected_features_pvals)
    print(f"P-value selected features saved to: {output_path_pvals}")

    # ------------------------------------------------------------------
    # 2. Feature Selection using Lasso Regularization (as per paper)
    #    - Tunes Lasso coefficient on train split.
    #    - Selects features with non-zero coefficients.
    #    - Aims for ~78 features (the paper mentions 78 features).
    #    - Best performance over validation split is chosen.
    # ------------------------------------------------------------------
    print("\n--- Lasso Regularization based Feature Selection ---")

    # Define a range of alpha values for Lasso. Start with a broader range
    # and then refine if needed. Logarithmic scale is often good.
    # The goal is to find an alpha that yields ~78 non-zero features AND good R2.
    alpha_values = np.logspace(-4, 0, 50)  # Example range, adjust as needed

    best_r2_lasso = -np.inf
    selected_features_lasso = []
    num_lasso_features = 0

    for alpha in alpha_values:
        lasso_model = Lasso(alpha=alpha, max_iter=2000, random_state=42)
        lasso_model.fit(X_train, y_train)

        # Get features with non-zero coefficients
        current_lasso_features = X_train.columns[lasso_model.coef_ != 0].tolist()
        current_num_features = len(current_lasso_features)

        current_r2_lasso = -np.inf  # initialise before conditional to avoid NameError
        if current_num_features > 0:  # Ensure at least one feature is selected
            # Train a Linear Regression model with the selected Lasso features
            # (Using LinearRegression for R2 evaluation as per the paper implies
            # comparing the *set of features* rather than Lasso's direct prediction)
            lr_model_for_r2 = LinearRegression()
            lr_model_for_r2.fit(X_train[current_lasso_features], y_train)
            y_val_pred_lasso = lr_model_for_r2.predict(X_val[current_lasso_features])
            # current_r2_lasso = r2_score(y_val, y_val_pred_lasso)
            current_r2_lasso = adjusted_r2(
                y_val, y_val_pred_lasso, current_num_features
            )

            # We are looking for the best R2, and the paper specifically mentions ~78 features
            # So we can prioritize R2 while keeping an eye on the number of features.
            # This simple selection prioritizes R2. A more complex approach might try to get
            # closer to 78 features *while* having a good R2.
            if current_r2_lasso > best_r2_lasso:
                best_r2_lasso = current_r2_lasso
                selected_features_lasso = current_lasso_features
                num_lasso_features = current_num_features

        print(
            f"  Alpha={alpha:.6f}, Features={current_num_features}, R2={current_r2_lasso:.4f}"
        )  # Uncomment for detailed progress

    print(
        f"Best R2 for Lasso selection on validation: {best_r2_lasso:.4f} with {num_lasso_features} features."
    )
    print(f"Number of Lasso selected features: {len(selected_features_lasso)}")
    print("Selected Lasso features (first 5):", selected_features_lasso[:5])

    # Save the selected features
    output_path_lasso = os.path.join(DATA_DIR, "selected_features_lasso.npy")
    np.save(output_path_lasso, selected_features_lasso)
    print(f"Lasso selected features saved to: {output_path_lasso}")

    # Save R² comparison
    lr_all = LinearRegression().fit(X_train, y_train)
    r2_all = adjusted_r2(y_val, lr_all.predict(X_val), X_train.shape[1])

    r2_json = {
        "none": float(r2_all),
        "pvalue": float(best_r2_pvals),
        "lasso": float(best_r2_lasso),
    }

    with open(_r2_path, "w") as f:
        json.dump(r2_json, f, indent=2)

    print(f"Feature selection R² saved to: {_r2_path}")
    print(
        f"None={r2_all:.4f} | P-value={best_r2_pvals:.4f} | Lasso={best_r2_lasso:.4f}"
    )

    print("\nFeature selection process complete. Selected feature sets are saved.")

---
## MODULE 4 — Model Training
**Inputs:** split CSVs + `selected_features_lasso.npy` from MODULE 3  
**Outputs:** `results_metrics.csv`, `best_params.json`, `trained_models/*.joblib`  
Requires GPU (cuML). Skip if `results_metrics.csv` already exists.

In [ ]:
!pip uninstall -y \
  cudf-cu12 cuml-cu12 rmm-cu12 pylibcudf-cu12 cuvs-cu12 \
  pylibcugraph-cu12 libcugraph-cu12 cudf-polars-cu12 pylibraft-cu12 libraft-cu12

!pip install \
  cudf-cu12==25.02.* \
  cuml-cu12==25.02.* \
  rmm-cu12==25.02.* \
  pylibcudf-cu12==25.02.* \
  cuvs-cu12==25.02.* \
  --extra-index-url=https://pypi.nvidia.com -q

In [ ]:
!nvcc --version
!nvidia-smi

In [ ]:
# Wait for background cuML install to complete
if "install_done" in dir():
    install_done.wait()
print("Importing cuML...")
import cupy as cp
from cuml.linear_model import Ridge as cuRidge, LinearRegression as cuLinearRegression
from cuml.svm import SVR as cuSVR
from cuml.cluster import KMeans as cuKMeans
import xgboost as xgb
import optuna
import numpy as np, pandas as pd, os, json
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import optimizers
import joblib

optuna.logging.set_verbosity(optuna.logging.INFO)

os.makedirs(os.path.join(MODELS_DIR, "trained_models"), exist_ok=True)

N_TRIALS = 30
NUM_ITERATIONS = 300
BATCH_SIZE = 256

# Sentinel: skip if already done
_metrics_path = os.path.join(MODELS_DIR, "results_metrics.csv")
if os.path.exists(_metrics_path):
    print("✅ results_metrics.csv already exists — skipping training.")
    print("   Delete it to force re-training.")
else:
    print("Loading data...")
    X_train = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_train_X.csv"))
    y_train = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_train_y.csv")).iloc[:, 0]
    X_val = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_val_X.csv"))
    y_val = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_val_y.csv")).iloc[:, 0]
    X_test = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_test_X.csv"))
    y_test = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_test_y.csv")).iloc[:, 0]
    selected_features_lasso = np.load(
        os.path.join(DATA_DIR, "selected_features_lasso.npy"), allow_pickle=True
    ).tolist()

    X_train_lasso = X_train[selected_features_lasso]
    X_val_lasso = X_val[selected_features_lasso]
    X_test_lasso = X_test[selected_features_lasso]
    print(f"Data loaded. Lasso features: {len(selected_features_lasso)}")

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
print("Waiting for cuML install...")
if "install_done" in dir():
    install_done.wait()  # blocks until done, no-op if already finished
else:
    print("Warning: install_done not defined — MODULE 0 was not run in this session.")
print("Done.")


# import cupy as cp
# from cuml.linear_model import Ridge as cuRidge, LinearRegression as cuLinearRegression
# from cuml.svm import SVR as cuSVR
# from cuml.cluster import KMeans as cuKMeans
from cuml.metrics import r2_score as cu_r2_score
import xgboost as xgb  # GPU Gradient Boosting

import cupy as cp
from cuml.linear_model import Ridge as cuRidge, LinearRegression as cuLinearRegression
from cuml.svm import SVR as cuSVR
from cuml.cluster import KMeans as cuKMeans

print("✅ cuML OK")
gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"]
if isinstance(gpu_name, bytes):
    gpu_name = gpu_name.decode()
print(f"GPU: {gpu_name}")

import optuna
import numpy as np
import pandas as pd
import os
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from tensorflow.keras.optimizers import Adam

N_TRIALS = 30
NUM_ITERATIONS = 300
BATCH_SIZE = 256


# Helper: convert DataFrame/Series to cupy array
def to_gpu(X):
    if isinstance(X, pd.DataFrame) or isinstance(X, pd.Series):
        return cp.array(X.values, dtype=cp.float32)
    return cp.array(X, dtype=cp.float32)


# --- helpers ---


def compute_regression_metrics(y_true, y_pred):
    # Convert to CPU numpy arrays for sklearn metrics
    y_true_np = (
        cp.asnumpy(y_true) if isinstance(y_true, cp.ndarray) else np.array(y_true)
    )
    y_pred_np = (
        cp.asnumpy(y_pred) if isinstance(y_pred, cp.ndarray) else np.array(y_pred)
    )
    return {
        "R2": r2_score(y_true_np, y_pred_np),
        "MSE": mean_squared_error(y_true_np, y_pred_np),
        "MAE": mean_absolute_error(y_true_np, y_pred_np),
    }


def retrain_and_eval(
    model, X_train, y_train, X_val, y_val, X_test, y_test, name, best_val_r2=None
):
    X_tv = pd.concat([X_train, X_val], ignore_index=True)
    y_tv = pd.concat([y_train, y_val], ignore_index=True).values.ravel()

    X_tv_gpu = to_gpu(X_tv)
    y_tv_gpu = to_gpu(y_tv)
    X_te_gpu = to_gpu(X_test)
    y_te_np = y_test.values if isinstance(y_test, pd.Series) else np.array(y_test)

    model.fit(X_tv_gpu, y_tv_gpu)

    pred_tv = model.predict(X_tv_gpu)
    pred_test = model.predict(X_te_gpu)

    trainval_metrics = compute_regression_metrics(y_tv_gpu, pred_tv)
    test_metrics = compute_regression_metrics(y_te_np, pred_test)

    return {
        "trainval_R2": trainval_metrics["R2"],
        "trainval_MSE": trainval_metrics["MSE"],
        "trainval_MAE": trainval_metrics["MAE"],
        "test_R2": test_metrics["R2"],
        "test_MSE": test_metrics["MSE"],
        "test_MAE": test_metrics["MAE"],
        "best_R2_tuning": best_val_r2,
    }


# ── 0. Baseline Linear ─────────────────────────────────────────────────────────


def LinearModel(X_train, y_train, X_val, y_val, X_test, y_test):
    model = cuLinearRegression()
    metrics = retrain_and_eval(
        model, X_train, y_train, X_val, y_val, X_test, y_test, "Linear Regression"
    )
    print(
        f"[Linear Baseline] trainval R2={metrics['trainval_R2']:.4f} | test R2={metrics['test_R2']:.4f}"
    )
    return model, metrics


# ── 1. Ridge ───────────────────────────────────────────────────────────────────


def LinearModelRidge(X_train, y_train, X_val, y_val, X_test, y_test):
    X_tr_gpu = to_gpu(X_train)
    y_tr_gpu = to_gpu(y_train)
    X_v_gpu = to_gpu(X_val)
    y_v_np = y_val.values

    def objective(trial):
        alpha = trial.suggest_float("alpha", 1e-3, 1e3, log=True)
        m = cuRidge(alpha=alpha)
        m.fit(X_tr_gpu, y_tr_gpu)
        pred = cp.asnumpy(m.predict(X_v_gpu))
        return r2_score(y_v_np, pred)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best = study.best_params
    print(f"[Ridge] best params={best}, val R2={study.best_value:.4f}")

    model = cuRidge(alpha=best["alpha"])
    return model, retrain_and_eval(
        model, X_train, y_train, X_val, y_val, X_test, y_test, "Ridge", study.best_value
    )


# ── 2. Gradient Boosting → XGBoost GPU ────────────────────────────────────────


def TreebasedModel(X_train, y_train, X_val, y_val, X_test, y_test):
    # XGBoost accepte directement les DataFrames
    y_tr_np = y_train.values.ravel()
    y_v_np = y_val.values.ravel()

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "tree_method": "hist",
            "device": "cuda",  # ← GPU
            "random_state": 42,
        }
        m = xgb.XGBRegressor(**params, verbosity=0)
        m.fit(X_train, y_tr_np)
        return r2_score(y_v_np, m.predict(X_val))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best = study.best_params
    print(f"[XGBoost] best params={best}, val R2={study.best_value:.4f}")

    model = xgb.XGBRegressor(
        **best, tree_method="hist", device="cuda", random_state=42, verbosity=0
    )
    # XGBoost gère nativement pandas, pas besoin de to_gpu dans retrain_and_eval
    # → on override manuellement ici
    X_tv = pd.concat([X_train, X_val], ignore_index=True)
    y_tv = pd.concat([y_train, y_val], ignore_index=True).values.ravel()
    model.fit(X_tv, y_tv)

    pred_tv = model.predict(X_tv)
    pred_test = model.predict(X_test)
    y_te_np = y_test.values

    return model, {
        "trainval_R2": r2_score(y_tv, pred_tv),
        "trainval_MSE": mean_squared_error(y_tv, pred_tv),
        "trainval_MAE": mean_absolute_error(y_tv, pred_tv),
        "test_R2": r2_score(y_te_np, pred_test),
        "test_MSE": mean_squared_error(y_te_np, pred_test),
        "test_MAE": mean_absolute_error(y_te_np, pred_test),
        "best_R2_tuning": study.best_value,
    }


# ── 3. KMeans + Ridge ──────────────────────────────────────────────────────────


def kmeans(X_train, y_train, X_val, y_val, X_test, y_test):
    X_tr_gpu = to_gpu(X_train)
    y_tr_np = y_train.values.ravel()
    X_v_gpu = to_gpu(X_val)
    y_v_np = y_val.values.ravel()

    def objective(trial):
        n_clusters = trial.suggest_int("n_clusters", 4, 16)
        alpha = trial.suggest_float("alpha", 1e-2, 1e3, log=True)

        km = cuKMeans(n_clusters=n_clusters, random_state=0)
        km.fit(X_tr_gpu)
        c_tr = cp.asnumpy(km.predict(X_tr_gpu))
        c_v = cp.asnumpy(km.predict(X_v_gpu))

        preds = np.zeros(len(y_v_np))
        for i in range(n_clusters):
            mask_tr = c_tr == i
            mask_v = c_v == i
            if mask_tr.sum() < 2 or mask_v.sum() == 0:
                continue
            r = cuRidge(alpha=alpha)
            r.fit(to_gpu(X_train.values[mask_tr]), to_gpu(y_tr_np[mask_tr]))
            preds[mask_v] = cp.asnumpy(r.predict(to_gpu(X_val.values[mask_v])))
        return r2_score(y_v_np, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best = study.best_params
    print(f"[KMeans+Ridge] best params={best}, val R2={study.best_value:.4f}")

    X_tv_gpu = to_gpu(pd.concat([X_train, X_val], ignore_index=True))
    y_tv_np = pd.concat([y_train, y_val], ignore_index=True).values.ravel()
    X_te_gpu = to_gpu(X_test)
    y_te_np = y_test.values
    n_clusters = best["n_clusters"]
    alpha = best["alpha"]

    km_final = cuKMeans(n_clusters=n_clusters, random_state=0)
    km_final.fit(X_tv_gpu)
    c_tv = cp.asnumpy(km_final.predict(X_tv_gpu))
    c_te = cp.asnumpy(km_final.predict(X_te_gpu))

    preds_tv = np.zeros(len(y_tv_np))
    preds_te = np.zeros(len(y_te_np))
    for i in range(n_clusters):
        mask_tv = c_tv == i
        mask_te = c_te == i
        if mask_tv.sum() < 2:
            continue
        r = cuRidge(alpha=alpha)
        r.fit(X_tv_gpu[mask_tv], to_gpu(y_tv_np[mask_tv]))
        if mask_tv.sum() > 0:
            preds_tv[mask_tv] = cp.asnumpy(r.predict(X_tv_gpu[mask_tv]))
        if mask_te.sum() > 0:
            preds_te[mask_te] = cp.asnumpy(r.predict(X_te_gpu[mask_te]))

    return km_final, {
        "trainval_R2": r2_score(y_tv_np, preds_tv),
        "trainval_MSE": mean_squared_error(y_tv_np, preds_tv),
        "trainval_MAE": mean_absolute_error(y_tv_np, preds_tv),
        "test_R2": r2_score(y_te_np, preds_te),
        "test_MSE": mean_squared_error(y_te_np, preds_te),
        "test_MAE": mean_absolute_error(y_te_np, preds_te),
        "best_R2_tuning": study.best_value,
    }


# ── 4. SVR ─────────────────────────────────────────────────────────────────────


def svm_model(X_train, y_train, X_val, y_val, X_test, y_test):
    X_tr_gpu = to_gpu(X_train)
    y_tr_gpu = to_gpu(y_train)
    X_v_gpu = to_gpu(X_val)
    y_v_np = y_val.values

    def objective(trial):
        C = trial.suggest_float("C", 1e-1, 1e3, log=True)
        gamma = trial.suggest_float("gamma", 1e-4, 1e0, log=True)
        m = cuSVR(kernel="rbf", C=C, gamma=gamma)
        m.fit(X_tr_gpu, y_tr_gpu)
        pred = cp.asnumpy(m.predict(X_v_gpu))
        return r2_score(y_v_np, pred)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best = study.best_params
    print(f"[SVR] best params={best}, val R2={study.best_value:.4f}")

    model = cuSVR(kernel="rbf", C=best["C"], gamma=best["gamma"])
    return model, retrain_and_eval(
        model, X_train, y_train, X_val, y_val, X_test, y_test, "SVR", study.best_value
    )


# --- Main Execution Block ---
if __name__ == "__main__":
    # Corrected File paths for input data (normalized X, raw y)
    train_x_path = os.path.join(DATA_DIR, "data_cleaned_train_X.csv")
    train_y_path = os.path.join(DATA_DIR, "data_cleaned_train_y.csv")
    val_x_path = os.path.join(DATA_DIR, "data_cleaned_val_X.csv")
    val_y_path = os.path.join(DATA_DIR, "data_cleaned_val_y.csv")
    test_x_path = os.path.join(DATA_DIR, "data_cleaned_test_X.csv")
    test_y_path = os.path.join(DATA_DIR, "data_cleaned_test_y.csv")

    # Load the normalized feature data and non-normalized target data
    X_train_norm = pd.read_csv(train_x_path)
    y_train_raw = pd.read_csv(train_y_path)
    X_val_norm = pd.read_csv(val_x_path)
    y_val_raw = pd.read_csv(val_y_path)
    X_test_norm = pd.read_csv(test_x_path)
    y_test_raw = pd.read_csv(test_y_path)

    y_train_raw = y_train_raw.squeeze("columns")
    y_val_raw = y_val_raw.squeeze("columns")
    y_test_raw = y_test_raw.squeeze("columns")

    print("Data loaded for model training.")
    print(
        f"X_train_norm shape: {X_train_norm.shape}, y_train_raw shape: {y_train_raw.shape}"
    )
    print(f"X_val_norm shape: {X_val_norm.shape}, y_val_raw shape: {y_val_raw.shape}")
    print(
        f"X_test_norm shape: {X_test_norm.shape}, y_test_raw shape: {y_test_raw.shape}"
    )

    # --- BASELINE: Linear Regression with ALL features ---
    print(
        "\n-------------------- Linear Regression (Baseline - All Features) --------------------\n"
    )
    results_metrics = {}

    _linear_model_obj, results_metrics["Linear Baseline"] = LinearModel(
        X_train_norm,
        y_train_raw,
        X_val_norm,
        y_val_raw,
        X_test_norm,
        y_test_raw,
    )

    # --- Apply Lasso Feature Selection ---
    print(
        "\n-------------------- Applying Lasso Feature Selection --------------------\n"
    )
    lasso_features_path = os.path.join(DATA_DIR, "selected_features_lasso.npy")
    try:
        selected_features_lasso = np.load(
            lasso_features_path, allow_pickle=True
        ).tolist()
        if not selected_features_lasso:
            print(
                "Warning: Lasso selected features list is empty. Falling back to all features."
            )
            selected_features_lasso = X_train_norm.columns.tolist()
    except FileNotFoundError:
        print(f"Error: Lasso selected features file not found at {lasso_features_path}")
        selected_features_lasso = X_train_norm.columns.tolist()
        print("Proceeding with all features as a fallback for subsequent models.")
    except Exception as e:
        print(f"An unexpected error occurred while loading Lasso features: {e}")
        selected_features_lasso = X_train_norm.columns.tolist()
        print("Proceeding with all features as a fallback for subsequent models.")

    # Filter datasets using Lasso selected features
    if len(selected_features_lasso) > 0:
        X_train_lasso = X_train_norm[selected_features_lasso]
        y_train_lasso = y_train_raw
        X_val_lasso = X_val_norm[selected_features_lasso]
        y_val_lasso = y_val_raw
        X_test_lasso = X_test_norm[selected_features_lasso]
        y_test_lasso = y_test_raw

        X_concat_lasso = pd.concat([X_train_lasso, X_val_lasso], ignore_index=True)
        y_concat_lasso = pd.concat([y_train_lasso, y_val_lasso], ignore_index=True)

        print(
            f"Data filtered using {len(selected_features_lasso)} Lasso selected features."
        )
        print(
            f"X_concat_lasso shape: {X_concat_lasso.shape}, y_concat_lasso shape: {y_concat_lasso.shape}"
        )
        print(
            f"X_test_lasso shape: {X_test_lasso.shape}, y_test_lasso shape: {y_test_lasso.shape}"
        )

        # 1. Ridge Regression
        print("\n-------------------- Ridge Regression --------------------\n")
        _ridge_model_obj, results_metrics["Ridge"] = LinearModelRidge(
            X_train_lasso,
            y_train_lasso,
            X_val_lasso,
            y_val_lasso,
            X_test_lasso,
            y_test_lasso,
        )

        # 2. Gradient Boosting Tree Ensemble
        print(
            "\n-------------------- Gradient Boosting Tree Ensemble --------------------\n"
        )
        _gb_model_obj, results_metrics["Gradient Boosting"] = TreebasedModel(
            X_train_lasso,
            y_train_lasso,
            X_val_lasso,
            y_val_lasso,
            X_test_lasso,
            y_test_lasso,
        )

        # 3. K-means Clustering with Ridge Regression
        print(
            "\n-------------------- K-means Clustering with Ridge Regression --------------------\n"
        )
        _kmeans_model_obj, results_metrics["KMeans+Ridge"] = kmeans(
            X_train_lasso,
            y_train_lasso,
            X_val_lasso,
            y_val_lasso,
            X_test_lasso,
            y_test_lasso,
        )

        # 4. Support Vector Regression (SVR)
        print(
            "\n-------------------- Support Vector Regression (SVR) --------------------\n"
        )
        _svr_model_obj, results_metrics["SVR"] = svm_model(
            X_train_lasso,
            y_train_lasso,
            X_val_lasso,
            y_val_lasso,
            X_test_lasso,
            y_test_lasso,
        )

    else:
        print(
            "No Lasso features available. Skipping models that depend on Lasso features."
        )

Neural Network is in a different cell to avoid long running time for the first models.

In [ ]:
# ── 5. Neural Network ──────────────────────────────────────────────────────────
# TensorFlow uses GPU automatically; explicit numpy conversion avoids cupy/tf conflicts


def simple_neural_network(X_train, y_train, X_val, y_val, X_test, y_test):
    X_tr_np = X_train.values.astype(np.float32)
    y_tr_np = y_train.values.ravel().astype(np.float32)
    X_v_np = X_val.values.astype(np.float32)
    y_v_np = y_val.values.ravel().astype(np.float32)

    def build_model(trial, input_dim):
        units1 = trial.suggest_int("units1", 16, 128)
        units2 = trial.suggest_int("units2", 4, 64)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        m = Sequential(
            [
                Dense(units1, activation="relu", input_dim=input_dim),
                Dense(units2, activation="relu"),
                Dense(1, activation="linear"),
            ]
        )
        m.compile(loss="mse", optimizer=Adam(learning_rate=lr))
        return m

    def objective(trial):
        m = build_model(trial, X_tr_np.shape[1])
        m.fit(X_tr_np, y_tr_np, epochs=NUM_ITERATIONS, batch_size=BATCH_SIZE, verbose=0)
        return r2_score(y_v_np, m.predict(X_v_np).flatten())

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best = study.best_params
    print(f"[NN] best params={best}, val R2={study.best_value:.4f}")

    X_tv_np = np.vstack([X_tr_np, X_v_np])
    y_tv_np = np.concatenate([y_tr_np, y_v_np])
    final_model = build_model(optuna.trial.FixedTrial(best), X_tv_np.shape[1])
    final_model.fit(
        X_tv_np, y_tv_np, epochs=NUM_ITERATIONS, batch_size=BATCH_SIZE, verbose=0
    )

    pred_tv = final_model.predict(X_tv_np).flatten()
    pred_test = final_model.predict(X_test.values.astype(np.float32)).flatten()
    y_te_np = y_test.values

    return final_model, {
        "trainval_R2": r2_score(y_tv_np, pred_tv),
        "trainval_MSE": mean_squared_error(y_tv_np, pred_tv),
        "trainval_MAE": mean_absolute_error(y_tv_np, pred_tv),
        "test_R2": r2_score(y_te_np, pred_test),
        "test_MSE": mean_squared_error(y_te_np, pred_test),
        "test_MAE": mean_absolute_error(y_te_np, pred_test),
        "best_R2_tuning": study.best_value,
    }


print("\n-------------------- Neural Network --------------------\n")
_nn_model_obj, results_metrics["Neural Network"] = simple_neural_network(
    X_train_lasso,
    y_train_lasso,
    X_val_lasso,
    y_val_lasso,
    X_test_lasso,
    y_test_lasso,
)

print("\n========== R² Summary ==========")
for name, m in results_metrics.items():
    val_str = f"val R2={m['best_R2_tuning']:.4f} | " if m.get("best_R2_tuning") else ""
    print(
        f"  {name}: {val_str}test R2={m['test_R2']:.4f} | MSE={m['test_MSE']:.4f} | MAE={m['test_MAE']:.4f}"
    )


trained_models_dict = {
    "Linear Baseline": _linear_model_obj,
    "Ridge": _ridge_model_obj,
    "Gradient Boosting": _gb_model_obj,
    "KMeans+Ridge": _kmeans_model_obj,
    "SVR": _svr_model_obj,
    "Neural Network": _nn_model_obj,
}
print("✅ trained_models_dict construit")

In [ ]:
# Save all outputs
import pandas as pd, json, joblib, os

TM_DIR = os.path.join(MODELS_DIR, "trained_models")
os.makedirs(TM_DIR, exist_ok=True)

# 1. Metrics CSV
metrics_df = pd.DataFrame(results_metrics).T
metrics_df.index.name = "model"
metrics_df.to_csv(os.path.join(MODELS_DIR, "results_metrics.csv"))
print(f"✅ results_metrics.csv saved")


# 2. Best params JSON
def extract_params(mdl):
    if hasattr(mdl, "get_params"):
        return {
            k: (v.item() if hasattr(v, "item") else v)
            for k, v in mdl.get_params().items()
        }
    if hasattr(mdl, "get_config"):
        return mdl.get_config()
    return {}


best_params = {name: extract_params(mdl) for name, mdl in trained_models_dict.items()}
with open(os.path.join(MODELS_DIR, "best_params.json"), "w") as f:
    json.dump(best_params, f, indent=2, default=str)
print(f"✅ best_params.json saved")

# 3. Save trained models (joblib for sklearn-compatible; Keras saved separately)
for name, mdl in trained_models_dict.items():
    safe_name = name.replace("/", "_").replace(" ", "_").replace("+", "plus")
    path = os.path.join(TM_DIR, f"{safe_name}.joblib")
    try:
        joblib.dump(mdl, path)
        print(f"   Saved {name} → {path}")
    except Exception as e:
        print(f"   ⚠️  Could not joblib {name}: {e}")
        # Keras model fallback
        if hasattr(mdl, "save"):
            keras_path = os.path.join(TM_DIR, f"{safe_name}.keras")
            mdl.save(keras_path)
            print(f"   Saved Keras model → {keras_path}")

# 4. Save train+val lasso data for SHAP / arch modules
X_concat_lasso = pd.concat([X_train_lasso, X_val_lasso], ignore_index=True)
y_concat_full = pd.concat([y_train, y_val], ignore_index=True)
X_concat_lasso.to_csv(os.path.join(DATA_DIR, "data_trainval_lasso_X.csv"), index=False)
y_concat_full.to_csv(os.path.join(DATA_DIR, "data_trainval_lasso_y.csv"), index=False)
print("✅ train+val lasso arrays saved (for SHAP / arch modules)")

---
## MODULE 4b — Results Summary
**Inputs:** `results_metrics.csv`  

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, os

perf_df = pd.read_csv(
    os.path.join(MODELS_DIR, "results_metrics.csv"), index_col="model"
)

print("========== Model Performance Summary ==========")
print(
    perf_df[["test_R2", "test_MSE", "test_MAE", "trainval_R2"]].to_markdown(
        floatfmt=".4f"
    )
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#d9534f" if v < 0 else "#5b9bd5" for v in perf_df["test_R2"]]
bars = ax.bar(
    perf_df.index, perf_df["test_R2"], color=colors, edgecolor="black", alpha=0.85
)
ax.set_title("Test R² by Model", fontsize=13)
ax.set_ylabel("R²")
ax.set_ylim(perf_df["test_R2"].min() - 0.05, perf_df["test_R2"].max() + 0.05)
ax.tick_params(axis="x", rotation=20)
for bar, val in zip(bars, perf_df["test_R2"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f"{val:.4f}",
        ha="center",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

---
## MODULE 5 — SHAP Interpretability
**Inputs:** `trained_models/`, `data_trainval_lasso_X.csv`, `selected_features_lasso.npy`  
**Outputs:** SHAP plots (displayed inline)  

In [ ]:
import shap, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR as skSVR
from sklearn.metrics import r2_score

TM_DIR = os.path.join(MODELS_DIR, "trained_models")

# ── Load data ─────────────────────────────────────────────────────────────────
selected_features_lasso = np.load(
    os.path.join(DATA_DIR, "selected_features_lasso.npy"), allow_pickle=True
).tolist()
X_concat_lasso = pd.read_csv(os.path.join(DATA_DIR, "data_trainval_lasso_X.csv"))
y_concat_full = pd.read_csv(os.path.join(DATA_DIR, "data_trainval_lasso_y.csv")).iloc[
    :, 0
]
X_test_lasso = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_test_X.csv"))[
    selected_features_lasso
]


# ── Load models ───────────────────────────────────────────────────────────────
def load_model(name):
    safe = name.replace("/", "_").replace(" ", "_").replace("+", "plus")
    p = os.path.join(TM_DIR, f"{safe}.joblib")
    if os.path.exists(p):
        return joblib.load(p)
    from tensorflow.keras.models import load_model as keras_load

    kp = os.path.join(TM_DIR, f"{safe}.keras")
    if os.path.exists(kp):
        return keras_load(kp)
    raise FileNotFoundError(f"No saved model for {name}")


trained_models = {
    name: load_model(name)
    for name in [
        "Linear Baseline",
        "Ridge",
        "Gradient Boosting",
        "KMeans+Ridge",
        "SVR",
        "Neural Network",
    ]
}
print("✅ Models loaded")

# ── SHAP setup ────────────────────────────────────────────────────────────────
TOP_N = 20
N_SHAP = min(1000, len(X_concat_lasso))
X_shap = X_concat_lasso.sample(n=N_SHAP, random_state=42)


def plot_shap(shap_values, X_sample, title):
    shap.summary_plot(
        shap_values, X_sample, plot_type="bar", show=False, max_display=TOP_N
    )
    plt.title(f"SHAP Importance – {title}")
    plt.tight_layout()
    plt.show()
    shap.summary_plot(shap_values, X_sample, show=False, max_display=TOP_N)
    plt.title(f"SHAP Beeswarm – {title}")
    plt.tight_layout()
    plt.show()


def shap_summary_table(shap_values, X_sample, title, top_n=10):
    importance = (
        pd.DataFrame(
            {
                "feature": X_sample.columns,
                "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            }
        )
        .sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    importance.index += 1
    print(f"\n── Top {top_n} features by SHAP importance — {title} ──")
    print(importance.to_string())
    return importance


# ── Ridge ─────────────────────────────────────────────────────────────────────
print("--- SHAP: Ridge ---")
best_alpha = trained_models["Ridge"].alpha
ridge_cpu = Ridge(alpha=best_alpha).fit(X_concat_lasso, y_concat_full)
explainer = shap.LinearExplainer(ridge_cpu, X_shap)
sv_ridge = explainer.shap_values(X_shap)
plot_shap(sv_ridge, X_shap, "Ridge Regression")
shap_summary_table(sv_ridge, X_shap, "Ridge Regression")

# ── Gradient Boosting (XGBoost) ───────────────────────────────────────────────
print("--- SHAP: Gradient Boosting ---")
explainer = shap.TreeExplainer(trained_models["Gradient Boosting"])
sv_xgb = explainer.shap_values(X_shap)
plot_shap(sv_xgb, X_shap, "Gradient Boosting (XGBoost)")
shap_summary_table(sv_xgb, X_shap, "Gradient Boosting (XGBoost)")

# ── SVR ───────────────────────────────────────────────────────────────────────
print("--- SHAP: SVR (KernelExplainer — may take a few minutes) ---")
C = float(trained_models["SVR"].C)
gamma = float(trained_models["SVR"].gamma)

X_fit = X_concat_lasso.sample(n=2000, random_state=42)
y_fit = y_concat_full.loc[X_fit.index]

svr_cpu = skSVR(kernel="rbf", C=C, gamma=gamma)
svr_cpu.fit(X_fit, y_fit)
print(f"✅ SVR fitted — support vectors: {svr_cpu.n_support_.sum()}")

X_bg = shap.sample(X_fit, 30, random_state=42)
X_explain = X_fit.sample(50, random_state=0)

explainer = shap.KernelExplainer(svr_cpu.predict, X_bg)
sv_svr = explainer.shap_values(X_explain, nsamples=64)
plot_shap(sv_svr, X_explain, "SVR")
shap_summary_table(sv_svr, X_explain, "SVR")

# ── Neural Network ────────────────────────────────────────────────────────────
print("--- SHAP: Neural Network ---")
nn_model = trained_models["Neural Network"]
X_nn_np = X_shap.values.astype("float32")
X_bg_np = X_shap.sample(100, random_state=42).values.astype("float32")
explainer = shap.GradientExplainer(nn_model, X_bg_np)
sv_nn = explainer.shap_values(X_nn_np)[0]
plot_shap(sv_nn, X_shap, "Neural Network")
shap_summary_table(sv_nn, X_shap, "Neural Network")

# ── Synthèse cross-modèles ────────────────────────────────────────────────────
# Note: sv_svr est calculé sur X_explain (50 pts) — on aligne sur X_shap pour la synthèse
sv_svr_full = np.zeros((len(X_shap), X_shap.shape[1]))
sv_svr_full[X_explain.index.map(lambda i: X_shap.index.get_loc(i))] = sv_svr

all_importance = pd.DataFrame({"feature": X_shap.columns})
all_importance["Ridge"] = np.abs(sv_ridge).mean(axis=0)
all_importance["XGBoost"] = np.abs(sv_xgb).mean(axis=0)
all_importance["SVR"] = np.abs(sv_svr).mean(axis=0)  # mean sur les 50 pts explicatifs
all_importance["mean_rank"] = (
    all_importance[["Ridge", "XGBoost", "SVR"]].rank(ascending=False).mean(axis=1)
)
all_importance = all_importance.sort_values("mean_rank").head(15).reset_index(drop=True)
all_importance.index += 1
print("\n── Top 15 features par importance SHAP moyenne cross-modèles ──")
print(all_importance.to_string())

---
## MODULE 6 — Variance Assessment (Multi-seed)
**Inputs:** `data_cleaned.csv`, `selected_features_lasso.npy` (used as feature space reference only)  
**Outputs:** `variance_results.csv`

In [ ]:
import os, json, optuna, numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.svm import SVR  # ← CPU sklearn
from sklearn.cluster import KMeans  # ← CPU sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import time

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.INFO)

os.makedirs(MODELS_DIR, exist_ok=True)

N_RUNS = 5
RANDOM_SEEDS_FAST = [
    42,
    123,
    789,
    500,
    999,
    314,
    271,
    101,
    777,
    256,
]  # 10 seeds for fastest models
RANDOM_SEEDS_SLOW = [
    42,
    123,
    789,
    500,
    999,
    101,
]  # 6 seeds instead of the 5 intial to include 101 which was problematic for the baseline
BATCH_SIZE = 256
N_TRIALS = 20

MODEL_NAMES = [
    "Linear Baseline",
    "Ridge",
    "Gradient Boost",
    "KMeans+Ridge",
    "SVR",
    "Neural Net",
]
METRIC_KEYS = ["MAE_test", "MSE_test", "R2_test", "MAE_train", "MSE_train", "R2_train"]

# ── Helpers ───────────────────────────────────────────────────────────────────
# def to_gpu(X):
#     if hasattr(X, 'values'): return cp.array(X.values, dtype=cp.float32)
#     return cp.array(X, dtype=cp.float32)


def to_gpu(X):
    if hasattr(X, "values"):
        return X.values.astype(np.float32)
    return np.array(X, dtype=np.float32)


def split_data(dataset, val_frac=0.05, test_frac=0.05, random_state=None):
    X = dataset.drop(
        columns=[
            c for c in ["price", "id", "host_id", "Unnamed: 0"] if c in dataset.columns
        ]
    )
    y = dataset["price"]
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=val_frac + test_frac, random_state=random_state
    )
    X_test, X_val, y_test, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_frac / (val_frac + test_frac),
        random_state=random_state,
    )
    return X_train, y_train, X_val, y_val, X_test, y_test


def normalize_data(X_train, X_val, X_test):
    scaler = preprocessing.MinMaxScaler()
    return (
        pd.DataFrame(
            scaler.fit_transform(X_train.astype(float)), columns=X_train.columns
        ),
        pd.DataFrame(scaler.transform(X_val.astype(float)), columns=X_val.columns),
        pd.DataFrame(scaler.transform(X_test.astype(float)), columns=X_test.columns),
    )


def get_splits(seed):
    X_train, y_train, X_val, y_val, X_test, y_test = split_data(
        dataset_cleaned, random_state=seed
    )
    X_tr_n, X_val_n, X_te_n = normalize_data(X_train, X_val, X_test)
    X_concat = pd.concat([X_tr_n, X_val_n], ignore_index=True)
    y_concat = pd.concat([y_train, y_val], ignore_index=True)
    X_tr_l = X_tr_n[selected_features_lasso]
    X_val_l = X_val_n[selected_features_lasso]
    X_concat_l = X_concat[selected_features_lasso]
    X_te_l = X_te_n[selected_features_lasso]
    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        X_tr_l,
        X_val_l,
        X_concat_l,
        X_te_l,
        X_concat,
        y_concat,
    )


def evaluate_gpu(y_train, pred_train, y_test, pred_test):
    metrics = {
        "MAE_train": mean_absolute_error(y_train, pred_train),
        "MSE_train": mean_squared_error(y_train, pred_train),
        "R2_train": r2_score(y_train, pred_train),
        "MAE_test": mean_absolute_error(y_test, pred_test),
        "MSE_test": mean_squared_error(y_test, pred_test),
        "R2_test": r2_score(y_test, pred_test),
    }
    print(
        f"    → train R²={metrics['R2_train']:.4f} | test R²={metrics['R2_test']:.4f} | MAE={metrics['MAE_test']:.4f} | MSE={metrics['MSE_test']:.4f}"
    )
    return metrics


def save_model_cache(model_name):
    safe = model_name.replace("+", "plus").replace(" ", "_")
    path = os.path.join(MODELS_DIR, f"variance_{safe}.json")
    with open(path, "w") as f:
        json.dump(all_metrics[model_name], f, indent=2)
    print(f"✅ Sauvegardé → {path}")


def plot_results(summary_df):
    print("\n========== Variance Assessment – Test Metrics (mean ± std) ==========")
    print(
        summary_df[[c for c in summary_df.columns if "test" in c]].to_markdown(
            floatfmt=".4f"
        )
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(
        summary_df.index,
        summary_df["R2_test_mean"],
        yerr=summary_df["R2_test_std"],
        capsize=5,
        color="steelblue",
        edgecolor="black",
        alpha=0.8,
    )
    ax.set_title("Test R² – Mean ± Std across 5 random seeds")
    ax.set_ylabel("R²")
    ax.set_xlabel("Model")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


# Load data
dataset_cleaned = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned.csv"))
selected_features_lasso = np.load(
    os.path.join(DATA_DIR, "selected_features_lasso.npy"), allow_pickle=True
).tolist()
print(
    f"Dataset shape: {dataset_cleaned.shape} | {len(selected_features_lasso)} Lasso features"
)

# ── Load or init per-model cache ──────────────────────────────────────────────
all_metrics = {}
for model_name in MODEL_NAMES:
    safe = model_name.replace("+", "plus").replace(" ", "_")
    path = os.path.join(MODELS_DIR, f"variance_{safe}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            all_metrics[model_name] = json.load(f)
        print(
            f"✅ {model_name} chargé depuis cache ({len(all_metrics[model_name]['R2_test'])} seeds)"
        )
    else:
        all_metrics[model_name] = {k: [] for k in METRIC_KEYS}
        print(f"   {model_name} → pas de cache, initialisé vide")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BLOC 1 — Linear Baseline
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
print("\n===== Linear Baseline =====")
all_metrics["Linear Baseline"] = {k: [] for k in METRIC_KEYS}

for seed in RANDOM_SEEDS_FAST:
    print(f"  seed={seed}")
    X_train_raw, y_train_raw, _, _, X_test_raw, y_test_raw = split_data(
        dataset_cleaned, random_state=seed
    )
    lr_base = LinearRegression(n_jobs=-1).fit(X_train_raw, y_train_raw)
    m = evaluate_gpu(
        y_train_raw.values,
        lr_base.predict(X_train_raw),
        y_test_raw.values,
        lr_base.predict(X_test_raw),
    )
    for k, v in m.items():
        all_metrics["Linear Baseline"][k].append(v)

save_model_cache("Linear Baseline")
print(f"⏱ {(time.time() - t0) / 60:.1f} min")

# Print per-model summary
vals = all_metrics["Linear Baseline"]
print(f"\n  Linear Baseline — mean ± std over {len(vals['R2_test'])} seeds:")
print(f"  R²  test : {np.mean(vals['R2_test']):.4f} ± {np.std(vals['R2_test']):.4f}")
print(f"  MAE test : {np.mean(vals['MAE_test']):.4f} ± {np.std(vals['MAE_test']):.4f}")
print(f"  MSE test : {np.mean(vals['MSE_test']):.4f} ± {np.std(vals['MSE_test']):.4f}")
print(f"  R²  train: {np.mean(vals['R2_train']):.4f} ± {np.std(vals['R2_train']):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BLOC 2 — Ridge
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
print("\n===== Ridge =====")
all_metrics["Ridge"] = {k: [] for k in METRIC_KEYS}

for seed in RANDOM_SEEDS_FAST:
    print(f"  seed={seed}")
    (
        _,
        y_train,
        _,
        y_val,
        _,
        y_test,
        X_tr_l,
        X_val_l,
        X_concat_l,
        X_te_l,
        _,
        y_concat,
    ) = get_splits(seed)
    X_concat_gpu = to_gpu(X_concat_l)
    y_concat_gpu = to_gpu(y_concat)
    X_tr_gpu = to_gpu(X_tr_l)
    y_tr_gpu = to_gpu(y_train)
    X_v_gpu = to_gpu(X_val_l)
    y_v_np = y_val.values

    def objective(trial):
        m = cuRidge(alpha=trial.suggest_float("alpha", 1e-4, 1e3, log=True))
        m.fit(X_tr_gpu, y_tr_gpu)
        return r2_score(y_v_np, cp.asnumpy(m.predict(X_v_gpu)))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best_alpha = study.best_params["alpha"]
    print(f"    best_alpha={best_alpha:.4f}, val R²={study.best_value:.4f}")

    ridge = cuRidge(alpha=best_alpha)
    ridge.fit(X_concat_gpu, y_concat_gpu)
    m = evaluate_gpu(
        y_concat.values,
        cp.asnumpy(ridge.predict(X_concat_gpu)),
        y_test.values,
        cp.asnumpy(ridge.predict(to_gpu(X_te_l))),
    )
    for k, v in m.items():
        all_metrics["Ridge"][k].append(v)

save_model_cache("Ridge")
print(f"⏱ {(time.time() - t0) / 60:.1f} min")

vals = all_metrics["Ridge"]
print(f"\n  Ridge — mean ± std over {len(vals['R2_test'])} seeds:")
print(f"  R²  test : {np.mean(vals['R2_test']):.4f} ± {np.std(vals['R2_test']):.4f}")
print(f"  MAE test : {np.mean(vals['MAE_test']):.4f} ± {np.std(vals['MAE_test']):.4f}")
print(f"  MSE test : {np.mean(vals['MSE_test']):.4f} ± {np.std(vals['MSE_test']):.4f}")
print(f"  R²  train: {np.mean(vals['R2_train']):.4f} ± {np.std(vals['R2_train']):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BLOC 3 — Gradient Boosting
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
print("\n===== Gradient Boosting =====")
all_metrics["Gradient Boost"] = {k: [] for k in METRIC_KEYS}

for seed in RANDOM_SEEDS_FAST:
    print(f"  seed={seed}")
    (
        _,
        y_train,
        _,
        y_val,
        _,
        y_test,
        X_tr_l,
        X_val_l,
        X_concat_l,
        X_te_l,
        _,
        y_concat,
    ) = get_splits(seed)
    y_tr_np = y_train.values
    y_v_np = y_val.values

    def objective(trial):
        m = xgb.XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 500),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            tree_method="hist",
            device="cuda",
            random_state=seed,
            verbosity=0,
        )
        m.fit(X_tr_l, y_tr_np)
        return r2_score(y_v_np, m.predict(X_val_l))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"    best_params={study.best_params}, val R²={study.best_value:.4f}")

    X_tv = pd.concat([X_tr_l, X_val_l], ignore_index=True)
    y_tv = pd.concat([y_train, y_val], ignore_index=True).values
    gb = xgb.XGBRegressor(
        **study.best_params,
        tree_method="hist",
        device="cuda",
        random_state=seed,
        verbosity=0,
    )
    gb.fit(X_tv, y_tv)
    m = evaluate_gpu(y_tv, gb.predict(X_tv), y_test.values, gb.predict(X_te_l.values))
    for k, v in m.items():
        all_metrics["Gradient Boost"][k].append(v)

save_model_cache("Gradient Boost")
print(f"⏱ {(time.time() - t0) / 60:.1f} min")

vals = all_metrics["Gradient Boost"]
print(f"\n  Gradient Boost — mean ± std over {len(vals['R2_test'])} seeds:")
print(f"  R²  test : {np.mean(vals['R2_test']):.4f} ± {np.std(vals['R2_test']):.4f}")
print(f"  MAE test : {np.mean(vals['MAE_test']):.4f} ± {np.std(vals['MAE_test']):.4f}")
print(f"  MSE test : {np.mean(vals['MSE_test']):.4f} ± {np.std(vals['MSE_test']):.4f}")
print(f"  R²  train: {np.mean(vals['R2_train']):.4f} ± {np.std(vals['R2_train']):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BLOC 4 — KMeans+Ridge
# ══════════════════════════════════════════════════════════════════
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="cuml")
t0 = time.time()
print("\n===== KMeans+Ridge =====")
all_metrics["KMeans+Ridge"] = {k: [] for k in METRIC_KEYS}

for seed in RANDOM_SEEDS_FAST:
    print(f"  seed={seed}")
    (
        _,
        y_train,
        _,
        y_val,
        _,
        y_test,
        X_tr_l,
        X_val_l,
        X_concat_l,
        X_te_l,
        _,
        y_concat,
    ) = get_splits(seed)
    X_tr_gpu = to_gpu(X_tr_l)
    y_tr_np = y_train.values
    X_v_gpu = to_gpu(X_val_l)
    y_v_np = y_val.values

    def objective(trial):
        k = trial.suggest_int("n_clusters", 4, 16)
        alpha = trial.suggest_float("alpha", 1e-2, 1e3, log=True)
        km = cuKMeans(n_clusters=k, random_state=0)
        km.fit(X_tr_gpu)
        c_tr = cp.asnumpy(km.predict(X_tr_gpu))
        c_v = cp.asnumpy(km.predict(X_v_gpu))
        preds = np.zeros(len(y_v_np))
        for i in range(k):
            mtr, mv = c_tr == i, c_v == i
            if mtr.sum() < 2 or mv.sum() == 0:
                continue
            r = cuRidge(alpha=alpha)
            r.fit(to_gpu(X_tr_l.values[mtr]), to_gpu(y_tr_np[mtr]))
            preds[mv] = cp.asnumpy(r.predict(to_gpu(X_val_l.values[mv])))
        return r2_score(y_v_np, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    best_k = study.best_params["n_clusters"]
    best_alpha = study.best_params["alpha"]
    print(
        f"    best_k={best_k}, best_alpha={best_alpha:.4f}, val R²={study.best_value:.4f}"
    )

    X_concat_gpu = to_gpu(X_concat_l)
    y_concat_arr = y_concat.values
    km_f = cuKMeans(n_clusters=best_k, random_state=0)
    km_f.fit(X_concat_gpu)
    c_tv = cp.asnumpy(km_f.predict(X_concat_gpu))
    c_te = cp.asnumpy(km_f.predict(to_gpu(X_te_l)))
    y_te_all, yp_te_all, y_tr_all, yp_tr_all = [], [], [], []
    for i in range(best_k):
        mtr, mte = c_tv == i, c_te == i
        if mtr.sum() == 0 or mte.sum() == 0:
            continue
        r = cuRidge(alpha=best_alpha)
        r.fit(to_gpu(X_concat_l.values[mtr]), to_gpu(y_concat_arr[mtr]))
        y_te_all.extend(y_test.values[mte])
        yp_te_all.extend(cp.asnumpy(r.predict(to_gpu(X_te_l.values[mte]))))
        y_tr_all.extend(y_concat_arr[mtr])
        yp_tr_all.extend(cp.asnumpy(r.predict(to_gpu(X_concat_l.values[mtr]))))
    km_m = {
        "MAE_test": mean_absolute_error(y_te_all, yp_te_all),
        "MSE_test": mean_squared_error(y_te_all, yp_te_all),
        "R2_test": r2_score(y_te_all, yp_te_all),
        "MAE_train": mean_absolute_error(y_tr_all, yp_tr_all),
        "MSE_train": mean_squared_error(y_tr_all, yp_tr_all),
        "R2_train": r2_score(y_tr_all, yp_tr_all),
    }
    print(
        f"    → train R²={km_m['R2_train']:.4f} | test R²={km_m['R2_test']:.4f} | MAE={km_m['MAE_test']:.4f}"
    )
    for k, v in km_m.items():
        all_metrics["KMeans+Ridge"][k].append(v)

save_model_cache("KMeans+Ridge")
print(f"⏱ {(time.time() - t0) / 60:.1f} min")

vals = all_metrics["KMeans+Ridge"]
print(f"\n  KMeans+Ridge — mean ± std over {len(vals['R2_test'])} seeds:")
print(f"  R²  test : {np.mean(vals['R2_test']):.4f} ± {np.std(vals['R2_test']):.4f}")
print(f"  MAE test : {np.mean(vals['MAE_test']):.4f} ± {np.std(vals['MAE_test']):.4f}")
print(f"  MSE test : {np.mean(vals['MSE_test']):.4f} ± {np.std(vals['MSE_test']):.4f}")
print(f"  R²  train: {np.mean(vals['R2_train']):.4f} ± {np.std(vals['R2_train']):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BLOC 5 — SVR
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
print("\n===== SVR =====")
all_metrics["SVR"] = {k: [] for k in METRIC_KEYS}

for seed in RANDOM_SEEDS_SLOW:
    print(f"  seed={seed}")
    (
        _,
        y_train,
        _,
        y_val,
        _,
        y_test,
        X_tr_l,
        X_val_l,
        X_concat_l,
        X_te_l,
        _,
        y_concat,
    ) = get_splits(seed)
    X_tr_gpu = to_gpu(X_tr_l)
    y_tr_gpu = to_gpu(y_train)
    X_v_gpu = to_gpu(X_val_l)
    y_v_np = y_val.values
    X_concat_gpu = to_gpu(X_concat_l)
    y_concat_gpu = to_gpu(y_concat)

    def objective(trial):
        m = cuSVR(
            kernel="rbf",
            C=trial.suggest_float("C", 1e-1, 1e3, log=True),
            gamma=trial.suggest_float("gamma", 1e-4, 1e0, log=True),
        )
        m.fit(X_tr_gpu, y_tr_gpu)
        return r2_score(y_v_np, cp.asnumpy(m.predict(X_v_gpu)))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"    best_params={study.best_params}, val R²={study.best_value:.4f}")

    svr = cuSVR(kernel="rbf", **study.best_params)
    svr.fit(X_concat_gpu, y_concat_gpu)
    m = evaluate_gpu(
        y_concat.values,
        cp.asnumpy(svr.predict(X_concat_gpu)),
        y_test.values,
        cp.asnumpy(svr.predict(to_gpu(X_te_l))),
    )
    for k, v in m.items():
        all_metrics["SVR"][k].append(v)

save_model_cache("SVR")
print(f"⏱ {(time.time() - t0) / 60:.1f} min")

vals = all_metrics["SVR"]
print(f"\n  SVR — mean ± std over {len(vals['R2_test'])} seeds:")
print(f"  R²  test : {np.mean(vals['R2_test']):.4f} ± {np.std(vals['R2_test']):.4f}")
print(f"  MAE test : {np.mean(vals['MAE_test']):.4f} ± {np.std(vals['MAE_test']):.4f}")
print(f"  MSE test : {np.mean(vals['MSE_test']):.4f} ± {np.std(vals['MSE_test']):.4f}")
print(f"  R²  train: {np.mean(vals['R2_train']):.4f} ± {np.std(vals['R2_train']):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BLOC 6 — Neural Net
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
NUM_ITERATIONS = 50
print("\n===== Neural Net =====")
all_metrics["Neural Net"] = {k: [] for k in METRIC_KEYS}

for seed in RANDOM_SEEDS_SLOW:
    print(f"  seed={seed}")
    (
        _,
        y_train,
        _,
        y_val,
        _,
        y_test,
        X_tr_l,
        X_val_l,
        X_concat_l,
        X_te_l,
        _,
        y_concat,
    ) = get_splits(seed)
    X_tr_np = X_tr_l.values.astype(np.float32)
    y_tr_np = y_train.values.astype(np.float32)
    X_v_np = X_val_l.values.astype(np.float32)
    y_v_np = y_val.values.astype(np.float32)

    def objective(trial):
        m = Sequential(
            [
                Dense(
                    trial.suggest_int("units1", 16, 128),
                    activation="relu",
                    input_dim=X_tr_np.shape[1],
                ),
                Dense(trial.suggest_int("units2", 4, 64), activation="relu"),
                Dense(1, activation="linear"),
            ]
        )
        m.compile(
            loss="mse",
            optimizer=optimizers.Adam(
                learning_rate=trial.suggest_float("lr", 1e-4, 1e-2, log=True)
            ),
        )
        m.fit(X_tr_np, y_tr_np, epochs=NUM_ITERATIONS, batch_size=BATCH_SIZE, verbose=0)
        return r2_score(y_v_np, m.predict(X_v_np).flatten())

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"    best_params={study.best_params}, val R²={study.best_value:.4f}")

    X_tv_np = X_concat_l.values.astype(np.float32)
    y_tv_np = y_concat.values.astype(np.float32)
    nn = Sequential(
        [
            Dense(
                study.best_params["units1"],
                activation="relu",
                input_dim=X_tv_np.shape[1],
            ),
            Dense(study.best_params["units2"], activation="relu"),
            Dense(1, activation="linear"),
        ]
    )
    nn.compile(
        loss="mse", optimizer=optimizers.Adam(learning_rate=study.best_params["lr"])
    )
    nn.fit(X_tv_np, y_tv_np, epochs=NUM_ITERATIONS, batch_size=BATCH_SIZE, verbose=0)
    m = evaluate_gpu(
        y_tv_np,
        nn.predict(X_tv_np).flatten(),
        y_test.values,
        nn.predict(X_te_l.values.astype(np.float32)).flatten(),
    )
    for k, v in m.items():
        all_metrics["Neural Net"][k].append(v)

save_model_cache("Neural Net")
print(f"⏱ {(time.time() - t0) / 60:.1f} min")

vals = all_metrics["Neural Net"]
print(f"\n  Neural Net — mean ± std over {len(vals['R2_test'])} seeds:")
print(f"  R²  test : {np.mean(vals['R2_test']):.4f} ± {np.std(vals['R2_test']):.4f}")
print(f"  MAE test : {np.mean(vals['MAE_test']):.4f} ± {np.std(vals['MAE_test']):.4f}")
print(f"  MSE test : {np.mean(vals['MSE_test']):.4f} ± {np.std(vals['MSE_test']):.4f}")
print(f"  R²  train: {np.mean(vals['R2_train']):.4f} ± {np.std(vals['R2_train']):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BLOC 7 — Summary (après tous les blocs)
# ══════════════════════════════════════════════════════════════════
rows = []
for model_name in MODEL_NAMES:
    row = {"Model": model_name}
    for k in METRIC_KEYS:
        vals = all_metrics[model_name][k]
        row[f"{k}_mean"] = np.mean(vals) if vals else np.nan
        row[f"{k}_std"] = np.std(vals) if vals else np.nan
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Model")
plot_results(summary_df)

In [ ]:
# Save variance results
import pandas as pd, os

# `all_metrics` should be a dict {model_name: {seed: {metric: value}}} from cell above
# Flatten to DataFrame
rows = []
for model_name, seed_dict in all_metrics.items():
    for seed, m in seed_dict.items():
        rows.append({"model": model_name, "seed": seed, **m})
var_df = pd.DataFrame(rows)
var_df.to_csv(os.path.join(MODELS_DIR, "variance_results.csv"), index=False)
print(f"✅ variance_results.csv saved ({len(var_df)} rows)")

# Summary table
summary = var_df.groupby("model")[["R2_test", "MAE_test", "MSE_test"]].agg(
    ["mean", "std"]
)
print("\n========== Variance Summary ==========")
print(summary.to_markdown(floatfmt=".4f"))

---
## MODULE 7 — Architectural Exploration
**Inputs:** `data_trainval_lasso_X.csv`, `data_trainval_lasso_y.csv`, `data_cleaned_test_X/y.csv`, `selected_features_lasso.npy`  
**Outputs:** `arch_results.csv`

In [ ]:
import numpy as np
import pandas as pd
import os
import optuna
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras import optimizers
import warnings
from google.colab import drive

drive.mount("/content/drive")


optuna.logging.set_verbosity(optuna.logging.INFO)
warnings.filterwarnings("ignore")

selected_features_lasso = np.load(
    os.path.join(DATA_DIR, "selected_features_lasso.npy"), allow_pickle=True
).tolist()
X_concat_lasso = pd.read_csv(os.path.join(DATA_DIR, "data_trainval_lasso_X.csv"))
y_concat = pd.read_csv(os.path.join(DATA_DIR, "data_trainval_lasso_y.csv")).iloc[:, 0]
X_test_lasso = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_test_X.csv"))[
    selected_features_lasso
]
y_test = pd.read_csv(os.path.join(DATA_DIR, "data_cleaned_test_y.csv")).iloc[:, 0]

print(
    f"✅ Data loaded: X_train_val={X_concat_lasso.shape}, X_test={X_test_lasso.shape}"
)

In [ ]:
def report(name, y_true, y_pred):
    print(f"\n--- {name} ---")
    print(f"  R²:  {r2_score(y_true, y_pred):.4f}")
    print(f"  MAE: {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"  MSE: {mean_squared_error(y_true, y_pred):.4f}")
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
    }


arch_results = {}

# --- 3a. Optuna NN tuning ---
print("\n========== Optuna Neural Network Tuning ==========")
# Here we optimize the whole architecture, before we juste optimized nb of neurons in the fixed 2 layers


def build_tuned_nn(trial, input_dim):
    n_layers = trial.suggest_int("n_layers", 1, 4)
    model = Sequential()
    model.add(
        Dense(
            trial.suggest_int("units_0", 16, 256),
            activation="relu",
            input_dim=input_dim,
        )
    )
    if trial.suggest_categorical("batch_norm_0", [True, False]):
        model.add(BatchNormalization())
    model.add(Dropout(trial.suggest_float("dropout_0", 0.0, 0.5)))
    for i in range(1, n_layers):
        model.add(Dense(trial.suggest_int(f"units_{i}", 16, 128), activation="relu"))
        if trial.suggest_categorical(f"batch_norm_{i}", [True, False]):
            model.add(BatchNormalization())
        model.add(Dropout(trial.suggest_float(f"dropout_{i}", 0.0, 0.5)))
    model.add(Dense(1, activation="linear"))
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    model.compile(
        loss="mean_squared_error", optimizer=optimizers.Adam(learning_rate=lr)
    )
    return model


X_tr_opt, X_val_opt, y_tr_opt, y_val_opt = train_test_split(
    X_concat_lasso, y_concat, test_size=0.1, random_state=42
)


def objective(trial):
    model = build_tuned_nn(trial, X_concat_lasso.shape[1])
    bs = trial.suggest_categorical("batch_size", [64, 128, 256])
    model.fit(X_tr_opt, y_tr_opt, epochs=150, batch_size=bs, verbose=0)
    preds = model.predict(X_val_opt).ravel()
    return -r2_score(y_val_opt, preds)


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest Optuna params: {study.best_params}")
best_nn = build_tuned_nn(study.best_trial, X_concat_lasso.shape[1])
best_nn.fit(
    X_concat_lasso,
    y_concat,
    epochs=300,
    batch_size=study.best_params["batch_size"],
    verbose=0,
)
arch_results["Tuned NN (Optuna)"] = report(
    "Tuned NN (Optuna)", y_test, best_nn.predict(X_test_lasso).ravel()
)

# --- 3b. LightGBM + Optuna ---
print("\n========== LightGBM + Optuna ==========")


def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr_opt, y_tr_opt)
    return -r2_score(y_val_opt, m.predict(X_val_opt))


study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=30, show_progress_bar=True)

print(f"\nBest LightGBM params: {study_lgbm.best_params}")
lgbm = lgb.LGBMRegressor(
    **study_lgbm.best_params, random_state=42, n_jobs=-1, verbose=-1
)
lgbm.fit(X_concat_lasso, y_concat)
arch_results["LightGBM"] = report("LightGBM", y_test, lgbm.predict(X_test_lasso))


# --- 3c. Stacking Ensemble ---
print("\n========== Stacking Ensemble ==========")


def objective_ridge(trial):
    alpha = trial.suggest_float("alpha", 0.01, 100.0, log=True)
    m = Ridge(alpha=alpha)
    m.fit(X_tr_opt, y_tr_opt)
    return -r2_score(y_val_opt, m.predict(X_val_opt))


def objective_lgbm2(
    trial,
):  # Went for LightGBM 2 instead of a GradBoost due to computational constraints
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr_opt, y_tr_opt)
    return -r2_score(y_val_opt, m.predict(X_val_opt))


print("  Tuning Ridge...")
study_ridge = optuna.create_study(direction="minimize")
study_ridge.optimize(objective_ridge, n_trials=30, show_progress_bar=True)
print(f"  Best Ridge params: {study_ridge.best_params}")

print("  Tuning LightGBM_2...")
study_lgbm2 = optuna.create_study(direction="minimize")
study_lgbm2.optimize(objective_lgbm2, n_trials=30, show_progress_bar=True)
print(f"  Best LightGBM_2 params: {study_lgbm2.best_params}")

base_models = {
    "Ridge": Ridge(**study_ridge.best_params),
    "LightGBM_1": lgb.LGBMRegressor(
        **study_lgbm.best_params, random_state=42, n_jobs=-1, verbose=-1
    ),
    "LightGBM_2": lgb.LGBMRegressor(
        **study_lgbm2.best_params, random_state=42, n_jobs=-1, verbose=-1
    ),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
X_arr = X_concat_lasso.values
y_arr = y_concat.values
X_te = X_test_lasso.values

oof_preds = np.zeros((len(X_arr), len(base_models)))
test_preds = np.zeros((len(X_te), len(base_models)))

for j, (name, model) in enumerate(base_models.items()):
    test_fold_preds = np.zeros((len(X_te), kf.n_splits))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_arr)):
        m = model.__class__(**model.get_params())
        m.fit(X_arr[tr_idx], y_arr[tr_idx])
        oof_preds[val_idx, j] = m.predict(X_arr[val_idx])
        test_fold_preds[:, fold] = m.predict(X_te)
    test_preds[:, j] = test_fold_preds.mean(axis=1)
    print(f"  {name} OOF R²: {r2_score(y_arr, oof_preds[:, j]):.4f}")

meta = Ridge(alpha=1.0).fit(oof_preds, y_arr)
stacking_preds = meta.predict(test_preds)
arch_results["Stacking (Ridge meta)"] = report(
    "Stacking Ensemble", y_test, stacking_preds
)


# --- Final comparison table ---
print("\n========== Architectural Exploration – Summary ==========")
arch_df = pd.DataFrame(arch_results).T[["R2", "MAE", "MSE"]]
print(arch_df.to_markdown(floatfmt=".4f"))

In [ ]:
# ── Extract best configs from Optuna studies & fitted objects ─────────────────
best_configs = {
    "Tuned NN (Optuna)": study.best_params,
    "LightGBM": study_lgbm.best_params,
    "Stacking (Ridge meta)": {
        "Ridge": study_ridge.best_params,
        "LightGBM_1": study_lgbm.best_params,
        "LightGBM_2": study_lgbm2.best_params,
        "meta_learner": f"Ridge(alpha={meta.alpha})",
        "cv": "KFold(n_splits=5)",
    },
}

# ── Save ──────────────────────────────────────────────────────────────────────
arch_df = pd.DataFrame(arch_results).T[["R2", "MAE", "MSE"]]
arch_df.index.name = "model"
arch_df.to_csv(os.path.join(MODELS_DIR, "arch_results.csv"))

with open(os.path.join(MODELS_DIR, "arch_best_configs.json"), "w") as f:
    json.dump(best_configs, f, indent=2)

print("✅ arch_results.csv saved")
print("✅ arch_best_configs.json saved")
print(arch_df.to_markdown(floatfmt=".4f"))

### Random Seeds variance for LightGBM (best performing model (only a small improvement with stacking))

In [ ]:
import json

# Variance Assessment: LightGBM across seeds
print("\n===== LightGBM (Variance across seeds) =====")

SEEDS_LGBM = [42, 123, 789, 500, 999, 314, 271, 101, 777, 256]
lgbm_variance_results = []

for seed in SEEDS_LGBM:
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_concat_lasso, y_concat, test_size=0.1, random_state=seed
    )

    def objective_lgbm_var(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "random_state": seed,
            "n_jobs": -1,
            "verbose": -1,
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr)
        return -r2_score(y_val, m.predict(X_val))

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed)
    )
    study.optimize(objective_lgbm_var, n_trials=30, show_progress_bar=False)

    best_params = study.best_params
    model = lgb.LGBMRegressor(**best_params, random_state=seed, n_jobs=-1, verbose=-1)
    model.fit(X_concat_lasso, y_concat)

    train_preds = model.predict(X_concat_lasso)
    test_preds = model.predict(X_test_lasso)

    r2_train = r2_score(y_concat, train_preds)
    r2_test = r2_score(y_test, test_preds)
    mae_test = mean_absolute_error(y_test, test_preds)
    mse_test = mean_squared_error(y_test, test_preds)
    val_r2 = -study.best_value

    print(f"  seed={seed}")
    print(f"    best_params={best_params}, val R²={val_r2:.4f}")
    print(
        f"    → train R²={r2_train:.4f} | test R²={r2_test:.4f} | MAE={mae_test:.4f} | MSE={mse_test:.4f}"
    )

    lgbm_variance_results.append(
        {
            "seed": seed,
            "best_params": best_params,
            "val_r2": val_r2,
            "train_r2": r2_train,
            "test_r2": r2_test,
            "mae": mae_test,
            "mse": mse_test,
        }
    )

# Summary
r2s = [r["test_r2"] for r in lgbm_variance_results]
maes = [r["mae"] for r in lgbm_variance_results]
mses = [r["mse"] for r in lgbm_variance_results]
r2s_train = [r["train_r2"] for r in lgbm_variance_results]

print(f"\n  LightGBM — mean ± std over {len(SEEDS_LGBM)} seeds:")
print(f"  R²  test : {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
print(f"  MAE test : {np.mean(maes):.4f} ± {np.std(maes):.4f}")
print(f"  MSE test : {np.mean(mses):.4f} ± {np.std(mses):.4f}")
print(f"  R²  train: {np.mean(r2s_train):.4f} ± {np.std(r2s_train):.4f}")

# Save
out_path = os.path.join(MODELS_DIR, "variance_LightGBM.json")
with open(out_path, "w") as f:
    json.dump(lgbm_variance_results, f, indent=2)
print(f"✅ Sauvegardé → {out_path}")